## 1. ライブラリのインポートと設定

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score, average_precision_score
import lightgbm as lgb
from scipy import sparse

# 日本語フォント設定（必要に応じて）
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# スタイル設定
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 2. データの読み込み

分析結果ディレクトリを指定して、必要なファイルを読み込みます。

In [ ]:
# 分析結果ディレクトリを指定（最新のものを使用する場合は適宜変更）
result_dir = Path("results/experiments/feature_selection_prompt_last_token_20251126_135721")

# データディレクトリ
data_dir = result_dir / "data"

# メインデータの読み込み
feature_metrics = pd.read_csv(data_dir / "feature_metrics_full.csv")
candidates_suppress = pd.read_csv(data_dir / "candidates_suppress.csv")
candidates_amplify = pd.read_csv(data_dir / "candidates_amplify.csv")

print(f"全特徴数: {len(feature_metrics)}")
print(f"抑制候補数: {len(candidates_suppress)}")
print(f"増幅候補数: {len(candidates_amplify)}")
print("\n特徴メトリクスのカラム:")
print(feature_metrics.columns.tolist())

### データの概要確認

In [ ]:
# 基本統計量
print("=== 基本統計量 ===")
print(feature_metrics[['Log Ratio Syc/Base', 'Diff Base-Syc', 'SHAP Correlation', 'Intervention Score']].describe())

# 上位抑制候補
print("\n=== 上位抑制候補（Top 5） ===")
print(candidates_suppress.head())

# 上位増幅候補
print("\n=== 上位増幅候補（Top 5） ===")
print(candidates_amplify.head())

## 3. 特徴分布の可視化（Scatter Plot）

### 準備: 介入候補のマーキング

In [ ]:
# 介入候補のTop 20を取得
top_suppress_ids = set(candidates_suppress.head(20)['Feature ID'].values)
top_amplify_ids = set(candidates_amplify.head(20)['Feature ID'].values)

# マーカー用のカテゴリ列を追加
def categorize_feature(row):
    feature_id = row['Feature ID']
    if feature_id in top_suppress_ids:
        return 'Suppress (Top 20)'
    elif feature_id in top_amplify_ids:
        return 'Amplify (Top 20)'
    else:
        return 'Other'

feature_metrics['Category'] = feature_metrics.apply(categorize_feature, axis=1)

# カテゴリごとの数を確認
print(feature_metrics['Category'].value_counts())

### メイン散布図: Log Ratio vs Diff Base-Syc

In [ ]:
def plot_feature_distribution(df, top_suppress_ids, top_amplify_ids, figsize=(16, 10)):
    """
    特徴の分布を可視化する散布図を作成
    
    Parameters:
    -----------
    df : pd.DataFrame
        特徴メトリクスのデータフレーム
    top_suppress_ids : set
        抑制候補の特徴ID
    top_amplify_ids : set
        増幅候補の特徴ID
    figsize : tuple
        図のサイズ
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # データを3つのグループに分割
    df_other = df[df['Category'] == 'Other']
    df_suppress = df[df['Category'] == 'Suppress (Top 20)']
    df_amplify = df[df['Category'] == 'Amplify (Top 20)']
    
    # その他の特徴（背景）
    scatter1 = ax.scatter(
        df_other['Log Ratio Syc/Base'],
        df_other['Diff Base-Syc'],
        c=df_other['SHAP Correlation'],
        cmap='RdBu_r',
        alpha=0.3,
        s=20,
        vmin=-1,
        vmax=1,
        label='Other features'
    )
    
    # 抑制候補（目立たせる）
    scatter2 = ax.scatter(
        df_suppress['Log Ratio Syc/Base'],
        df_suppress['Diff Base-Syc'],
        c=df_suppress['SHAP Correlation'],
        cmap='RdBu_r',
        alpha=0.9,
        s=150,
        marker='D',
        edgecolors='black',
        linewidths=2,
        vmin=-1,
        vmax=1,
        label='Suppress candidates (Top 20)'
    )
    
    # 増幅候補（目立たせる）
    scatter3 = ax.scatter(
        df_amplify['Log Ratio Syc/Base'],
        df_amplify['Diff Base-Syc'],
        c=df_amplify['SHAP Correlation'],
        cmap='RdBu_r',
        alpha=0.9,
        s=150,
        marker='s',
        edgecolors='green',
        linewidths=2,
        vmin=-1,
        vmax=1,
        label='Amplify candidates (Top 20)'
    )
    
    # カラーバー
    cbar = plt.colorbar(scatter1, ax=ax)
    cbar.set_label('SHAP Correlation', fontsize=12)
    
    # 軸ラベルとタイトル
    ax.set_xlabel('Log Ratio Syc/Base (対数倍率)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Diff Base-Syc (強度差分)', fontsize=12, fontweight='bold')
    ax.set_title('SAE Feature Distribution: Intervention Candidates Highlighting', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # 凡例
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    
    # グリッド
    ax.grid(True, alpha=0.3)
    
    # 参照線（x=0, y=0）
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    
    # 領域の説明を追加
    ax.text(0.98, 0.98, 'Suppress Zone\n(右上)', 
            transform=ax.transAxes, fontsize=10, 
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.text(0.02, 0.02, 'Amplify Zone\n(左下)', 
            transform=ax.transAxes, fontsize=10,
            verticalalignment='bottom', horizontalalignment='left',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    plt.tight_layout()
    return fig, ax

# プロット実行
fig, ax = plot_feature_distribution(feature_metrics, top_suppress_ids, top_amplify_ids)
plt.show()

# 図を保存
fig.savefig(result_dir / "figures" / "feature_distribution_scatter.png", dpi=300, bbox_inches='tight')
print(f"図を保存しました: {result_dir / 'figures' / 'feature_distribution_scatter.png'}")

### 追加の可視化: SHAP Correlationの分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ヒストグラム
axes[0].hist(feature_metrics['SHAP Correlation'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero line')
axes[0].set_xlabel('SHAP Correlation', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of SHAP Correlation', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Intervention Scoreの分布
axes[1].hist(feature_metrics['Intervention Score'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Intervention Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Intervention Score', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 保存
fig.savefig(result_dir / "figures" / "score_distributions.png", dpi=300, bbox_inches='tight')
print(f"図を保存しました: {result_dir / 'figures' / 'score_distributions.png'}")

## 4. インタラクティブな特徴確認

特定の特徴IDの詳細情報を表示する関数

In [ ]:
def show_feature_details(feature_id, df):
    """
    指定された特徴IDの詳細情報を表示
    
    Parameters:
    -----------
    feature_id : int
        特徴ID
    df : pd.DataFrame
        特徴メトリクスのデータフレーム
    """
    feature = df[df['Feature ID'] == feature_id]
    
    if len(feature) == 0:
        print(f"特徴ID {feature_id} は見つかりませんでした。")
        return
    
    feature = feature.iloc[0]
    
    print("=" * 60)
    print(f"特徴ID: {feature_id}")
    print("=" * 60)
    print(f"カテゴリ: {feature['Category']}")
    print("\n--- 基本指標 ---")
    print(f"Activation Count (Base): {feature['Activation Count (Base)']:.0f}")
    print(f"Activation Count (Syc): {feature['Activation Count (Syc)']:.0f}")
    print(f"Mean Intensity (Base): {feature['Mean Intensity (Base)']:.6f}")
    print(f"Mean Intensity (Syc): {feature['Mean Intensity (Syc)']:.6f}")
    print("\n--- 重要な指標 ---")
    print(f"Log Ratio Syc/Base: {feature['Log Ratio Syc/Base']:.6f}")
    print(f"Diff Base-Syc: {feature['Diff Base-Syc']:.6f}")
    print(f"SHAP Correlation: {feature['SHAP Correlation']:.6f}")
    print(f"Specificity (Syc): {feature['Specificity (Syc)']:.6f}")
    print(f"Intervention Score: {feature['Intervention Score']:.6f}")
    print("=" * 60)

# 使用例: 上位抑制候補の1番目を表示
if len(candidates_suppress) > 0:
    top_suppress_id = candidates_suppress.iloc[0]['Feature ID']
    print("【抑制候補 Top 1】")
    show_feature_details(top_suppress_id, feature_metrics)

# 上位増幅候補の1番目を表示
if len(candidates_amplify) > 0:
    top_amplify_id = candidates_amplify.iloc[0]['Feature ID']
    print("\n【増幅候補 Top 1】")
    show_feature_details(top_amplify_id, feature_metrics)

In [ ]:
# 任意の特徴IDを指定して詳細を確認
# 以下の数値を変更して実行してください
custom_feature_id = 12345  # 確認したい特徴IDを指定
show_feature_details(custom_feature_id, feature_metrics)

## 5. モデル性能評価（ROC/PR曲線）

学習済みLightGBMモデルの性能を評価します。

### 5.1 データとモデルの準備

In [ ]:
# 元のJSONデータを読み込む（feature_selection.pyと同じデータを使用）
# ここでは最新のcombined_feedback_dataを使用する想定
json_path = Path("combined_feedback_data_v2.json")

print(f"データを読み込み中: {json_path}")
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# feature_selection.pyと同じロジックでデータを準備
def prepare_data_from_json(data):
    """
    JSONデータから特徴行列とラベルを抽出
    """
    features_list = []
    labels = []
    
    for result in data['results']:
        for variation in result['variations']:
            if variation['template_type'] == 'base':
                continue
            
            # SAE activationsを取得
            sae_activations = variation.get('sae_activations', {}).get('prompt_last_token', {})
            
            if not sae_activations:
                continue
            
            # 特徴ベクトルに変換
            feature_dict = {int(k): float(v) for k, v in sae_activations.items()}
            features_list.append(feature_dict)
            
            # ラベル
            label = variation.get('sycophancy_flag', 0)
            labels.append(label)
    
    return features_list, labels

features_list, labels = prepare_data_from_json(data)
print(f"サンプル数: {len(labels)}")
print(f"迎合性フラグ=1: {sum(labels)}")
print(f"迎合性フラグ=0: {len(labels) - sum(labels)}")

In [ ]:
# 疎行列に変換
from sklearn.feature_extraction import DictVectorizer

vectorizer = DictVectorizer()
X = vectorizer.fit_transform(features_list)
y = np.array(labels)

print(f"特徴行列の形状: {X.shape}")
print(f"ラベルの形状: {y.shape}")

In [ ]:
# Train/Test分割（feature_selection.pyと同じ設定）
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"訓練データ: {X_train.shape[0]} サンプル")
print(f"テストデータ: {X_test.shape[0]} サンプル")

In [ ]:
# LightGBMモデルの訓練（feature_selection.pyと同じパラメータ）
print("LightGBMモデルを訓練中...")

model = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

model.fit(X_train, y_train)
print("モデルの訓練が完了しました。")

### 5.2 ROC曲線とPR曲線の描画

In [ ]:
# 予測確率を取得
y_pred_proba = model.predict_proba(X_test)[:, 1]

# AUCスコアを計算
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f"ROC AUC: {roc_auc:.4f}")
print(f"PR AUC (Average Precision): {pr_auc:.4f}")

In [ ]:
# ROC曲線とPR曲線を横並びで描画
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ROC曲線
roc_display = RocCurveDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[0], name='LightGBM'
)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.50)')
axes[0].set_title(f'ROC Curve (AUC = {roc_auc:.4f})', fontsize=14, fontweight='bold')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# PR曲線
pr_display = PrecisionRecallDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[1], name='LightGBM'
)
# ベースライン（陽性率）
baseline = y_test.sum() / len(y_test)
axes[1].axhline(y=baseline, color='k', linestyle='--', 
                label=f'Baseline (Positive Rate = {baseline:.2f})')
axes[1].set_title(f'Precision-Recall Curve (AP = {pr_auc:.4f})', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 保存
fig.savefig(result_dir / "figures" / "model_performance_curves.png", dpi=300, bbox_inches='tight')
print(f"図を保存しました: {result_dir / 'figures' / 'model_performance_curves.png'}")

### 5.3 混同行列

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 予測ラベル（閾値0.5）
y_pred = (y_pred_proba >= 0.5).astype(int)

# 混同行列
cm = confusion_matrix(y_test, y_pred)

# 可視化
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
            xticklabels=['Non-Syc', 'Syc'],
            yticklabels=['Non-Syc', 'Syc'])
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 分類レポート
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Non-Syc', 'Syc']))

# 保存
fig.savefig(result_dir / "figures" / "confusion_matrix.png", dpi=300, bbox_inches='tight')
print(f"図を保存しました: {result_dir / 'figures' / 'confusion_matrix.png'}")

## 6. まとめ

このノートブックでは以下を実行しました:

1. **特徴分布の可視化**: Log Ratio vs Diff Base-Sycの散布図で、介入候補の位置を確認
2. **モデル性能評価**: ROC曲線、PR曲線、混同行列により、LightGBMモデルの性能を評価
3. **特徴詳細の確認**: 任意の特徴IDの詳細情報を表示する機能

### 次のステップ
- 特定された介入候補特徴を用いて、ステップ4の介入実験を実施
- 介入効果の評価と分析